# Sandbox for cleaning each table

TLDR:
All the cleaning will be summarised in functions which will be in a seperate notebook. 

This notebook is the raw process of how the cleaning was determined

## Imports & shared functions

In [0]:
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

In [0]:
%run ./utils

## Pin Topic

### Loading in data

In [0]:
df_pin = get_dataframe_from_drive('038444ac863e.pin')

#### Initial preview of data

In [0]:
display(df_pin.observe)
print(df_pin.count())

<bound method DataFrame.observe of DataFrame[category: string, description: string, downloaded: bigint, follower_count: string, image_src: string, index: bigint, is_image_or_video: string, poster_name: string, save_location: string, tag_list: string, title: string, unique_id: string]>

372


From the Observe function we can see the schema:
- category: string
- description: string
- downloaded: bigint
- follower_count: string
- image_src: string
- index: bigint
- is_image_or_video: string
- poster_name: string
- save_location: string
- tag_list: string
- title: string
- unique_id: string

In [0]:
display(df_pin.head(5))


category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id
christmas,"EASY Fingerprint Wreath Wall Art (With Your Family’s Fingerprints) Make your own boxwood wreath sign with fingerprints by following this easy, step-by-step tutorial and video! I…",1,71k,https://i.pinimg.com/videos/thumbnails/originals/32/f4/02/32f402be0b4d76c1775ad7d5d8a6c280.0000001.jpg,1899,video,"Karin Peters, Renovated Faith | Transforming Your Home & Heart",Local save in /data/christmas,"Christmas Decorations For Kids,Christmas Crafts For Adults,Diy Christmas Gifts For Family,Christmas On A Budget,Holiday Crafts,Christmas Time,Christmas Art Projects,Christmas Activities For Adults,Decorating For Christmas",DIY Fingerprint Wreath Wall Art (With Your Family’s Fingerprints & Name!),f0ddb462-8582-4d4c-9f19-426f652e712f
diy-and-crafts,"Grab your toilet paper rolls and make this pipe cleaner spider craft for kids! It's a fun Halloween project that is great for making with preschool, kindergarten, and elementary…",1,267k,https://i.pinimg.com/videos/thumbnails/originals/7d/70/2f/7d702f8214aaee92415cab35d8847432.0000001.jpg,2878,video,Easy Kids Crafts & Activities | Preschool & Kindergarten Ideas,Local save in /data/diy-and-crafts,"Halloween Arts And Crafts,Halloween Crafts For Toddlers,Fall Crafts For Kids,Toddler Crafts,Holiday Crafts,Halloween Activities For Preschoolers,Halloween Crafts For Kindergarten,Christmas Crafts For Kindergarteners,Arts And Crafts For Kids Easy",Pipe Cleaner Spider Craft For Kids,d2369cf7-7ed5-4080-abc0-fd5a1932e796
christmas,"6.5Ft + Pearl & Pine Cone Design - This Pearl & Pine Cone string light as a Christmas decor light. 6.5 Foot/2 Meters plated copper wire and 20 LED Lights, The led string lights…",1,5k,https://i.pinimg.com/originals/3f/94/40/3f94400a8d9878d76db9d68126161b13.jpg,2244,image,Wear24-7,Local save in /data/christmas,"Mini Christmas Tree Decorations,Christmas Pine Cones,Christmas Fairy Lights,Led Fairy Lights,Xmas Lights,Xmas Wreaths,Christmas Bells,Outdoor Christmas,Christmas Holidays",Christmas String Lights with Pine Cone Red Pearl Bell Garland with Lights 20 LED Warm White Battery Operated Christmas Tree Decor Light for Christmas Holiday Indoor Table Party Decoration,20485f66-1504-4272-bb96-049d7564abdd
event-planning,This fabulous DIY project made me drool when I first saw it and I knew immediately that I was going to have to make this! I absolutely love things like this...shiny sparkly thin…,1,985k,https://i.pinimg.com/originals/a6/79/3c/a6793c2e3deebca67ecd82b0087fc13c.jpg,4585,image,"DIY Joy - Crafts, Home Improvement, Decor & Recipes",Local save in /data/event-planning,"Cheap Favors,Wedding Favors Cheap,Wedding Invitations,Wedding Planning On A Budget,Event Planning,Wedding Table Decorations,Wedding Centerpieces,Dollar Tree Centerpieces,Centerpiece Ideas",She Attaches Crystals To A Plate And Creates A Breakfast At Tiffany's Inspired Item!,aa873546-701b-40dd-a339-a3f8aaf78ccb
home-decor,"TEAMThe latest in home decorating. Beautiful wall vinyl decals, that are simple to apply, are a great accent piece for any room, come in an array of colors, and are a cheap alte…",1,7k,https://i.pinimg.com/originals/57/d6/b4/57d6b49e5dda748308dd8852acdac3f1.jpg,6715,image,Boop Decals,Local save in /data/home-decor,"Office Wall Decor,Office Walls,Room Decor,Office Break Room,School Wall Decoration,Cheap Office Decor,School Office Decorations,Decorating Office At Work,Office Ideas For Work",Team Together Everyone Achieves More Quote Decal Sticker Wall Vinyl Art Home Room Decor Teacher School Classroom Science Work Office Job - orange,b94ad623-b1d3-457f-bc67-47991ce718a4


### Removing Duplicates

In [0]:
print(df_pin.count())
df_pin = df_pin.dropDuplicates(['unique_id'])
display(df_pin.count())

372


365

Here we can see that it has removed 7 duplicate rows.

### Detecting Null values

In [0]:
null_counts = get_null_counts_df(df_pin)
display(null_counts)

category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id
0,0,0,0,0,0,0,0,0,0,0,0


### Detecting Non-relevant Values (using Data Profile)

Each column may have different values that are null or have no relevant data 

In [0]:
display(df_pin)

category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id
vehicles,"The MRZR Diesel is an ultra-light turbo diesel combat vehicle perfectly suited for a wide range of tactical missions and certified for V-22, H-53, and H-47 internal air transport.",1,1k,https://i.pinimg.com/originals/de/fe/d6/defed65f30023145c4751f7e469334b9.jpg,10345,image,Michael Linderer,Local save in /data/vehicles,"Military Gear,Military Equipment,Army Vehicles,Armored Vehicles,Offroad,Polaris Off Road,Bug Out Vehicle,By Any Means Necessary,Polaris Ranger",Polaris MRZR Diesel | Polaris Government & Defense,00e1f3c5-ef2d-468c-91b1-b1bdc7ea8e37
tattoos,The foot is the sexiest part of the body and it is one of the best part of the body to have a tattoo. If you like to have a personal tattoo on your body then the foot is the bes…,1,873,https://i.pinimg.com/originals/36/e6/11/36e6116b96c87157b934dd9a4ea422e0.jpg,9050,image,Zbear,Local save in /data/tattoos,"Celtic Tattoo For Women,Simple Tattoos For Women,Ankle Tattoos For Women,Tattoos For Women Flowers,Celtic Tattoos,Flower Tattoo Meanings,Flower Tattoo Foot,Flower Tattoos,Tulip Tattoo",27 Outclass Foot Tattoos Ideas For Women,0103fb4b-a758-41da-a11b-234afa68505b
home-decor,CREATE A SPRING INSPIRED SOFA- Part of creating a beautiful home is giving a nod to the season! Sofas can help you breathe spring air into a room!,1,203k,https://i.pinimg.com/originals/49/2d/00/492d00fe990720051e93940e0b2d1daa.jpg,6349,image,StoneGable,Local save in /data/home-decor,"Coffee Table Styling,Decorating Coffee Tables,Coffee Table Tray Decor,Sofa Table Decor,Spring Home Decor,Diy Home Decor,Diy Décoration,Deco Table,Home Decor Inspiration",CREATE A SPRING INSPIRED SOFA,011b2462-ff5d-4b7c-bdeb-37781828e650
art,Shadow drawing is an awesome art process for young artists of any age. Considering shapes and positive & negative space are all part of making shadow art!,1,221k,https://i.pinimg.com/originals/6f/a6/ac/6fa6aca0168d8dee17c07c95210eb43d.png,341,image,The Kitchen Table Classroom,Local save in /data/art,"Shadow Drawing,Shadow Art,Middle School Art,Art School,High School,Programme D'art,Arte Elemental,Classe D'art,Art Curriculum",Shadow Drawing - An Experience in Positive and Negative Shapes - The Kitchen Table Classroom,0143fe12-3325-4e65-b138-f739df3cad77
art,"This fall tree silhouette art project is inspired by the beautiful leaves, as well as our animal friends that are scurrying around this time of year.",1,161k,https://i.pinimg.com/originals/02/c4/55/02c4558478c78e164ec2441fb15258fc.jpg,732,image,Make and Takes,Local save in /data/art,"September Art,September Crafts,Animal Art Projects,Fall Art Projects,Thanksgiving Art Projects,Kids Thanksgiving,Infant Art Projects,Halloween Art Projects,Art Projects",Fall Tree Silhouette Art Project - Make and Takes,02325753-68ad-4692-97a0-5b05bd513a5d
education,A list of more than 36 accommodations and strategies to address anxiety in your IEP or 504. #anxiety #GAD #IEP #school #education #IEPmeetingadvice,1,24k,https://i.pinimg.com/originals/d8/0f/2d/d80f2daa66493f1dc2f7616e5e2d98c1.jpg,3629,image,Fuzzymama | Simple ADHD Parenting Strategies,Local save in /data/education,"Gifted Education,Special Education Classroom,Physics Classroom,Kids Education,Physical Education,Special Education Organization,Texas Education,Education City,Classroom Board",{Anxiety} 43 Helpful IEP and 504 Plan Accommodations • A Day In Our Shoes,02a08753-216a-4107-b6cf-7051bf84a775
beauty,"Lymphatic drainage facial massage can improve uffiness, dull skin, acne, and even sensitivity. And doing it yourself is easier than you think!",1,6k,https://i.pinimg.com/originals/0f/88/3b/0f883b97c5014fe151b00d355800227e.png,1619,image,Renee Rouleau Skin Care,Local save in /data/beauty,"Beauty Care,Beauty Skin,Lymphatic Drainage Massage,Face Massage,Massage Tips,Face Yoga,Massage Techniques,Health And Beauty Tips,Beauty Tips For Skin",Lymphatic Drainage—

Databricks data profile. Run in Databricks to view.

By opening up the display functions Data profile compenent we are able to observe many insights like the distributions and null counts, and unique value counts. Any nulls or not related data will be replaced with `Nones`

Column:
- `downloaded` - is a numeric features it is confirmed to have only numeric values. However we can see that it has a Min of 0 and Max of 1, suggesting a boolean which can be confirmed upon ordering that column by descending showing no other values, hence it should be changed to a boolean datatype.
- `index` - col is a numeric features hence it is confirmed to have only numeric values.
- `category` - has 13 unique values, all of which are valid so no cleaning required.
- `description` - has 2 values which are not valid. "No description available Story format" & "No description available".
- `follower_count` appears to be a string column with numerical values, but some with k (thousands) or m (millions) as suffixes. These should be replaced so the whole column has only numerical values and be casted as such. Also has an invalid value of "User Info Error".
- `image_src` - valid values all have "https://i.pinimg.com/*" so will filter to only keep these rows. There was 1 invalid value found of "Image src error."
- `is_image_or_video` - has 3 distinct values all of which are valid so no cleaning required.
- `poster_name` - invalid values found of "User Info Error"
- `save_location` - all valid values have "Local save in *" so will filter to only keep these rows. No invalid values found
- `tag_list` - invalid value of "N,o, ,T,a,g,s, ,A,v,a,i,l,a,b,l,e".
- `title` - invalid value of "No Title Data Available"


#### Identifying non relavent values through functions instead of Data Profiler

Functions were created and stored in the utils file to identify mostly numeric values in string columns and a list of possibly invalid values

In [0]:
cols_to_ignore = ['unique_id', 'follower_count']
filter_mostly_numeric_strings(df_pin, threshold=0.5, exclude_cols=cols_to_ignore)

🔍 Potential Numeric Issues in Column: category


category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id


🔍 Potential Numeric Issues in Column: description


category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id


🔍 Potential Numeric Issues in Column: image_src


...

category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id


🔍 Potential Numeric Issues in Column: tag_list


category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id


🔍 Potential Numeric Issues in Column: title


category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id


In [0]:
from functools import reduce

In [0]:
invalid_values = [
    "none", "null", "n/a", "no data", "missing", "invalid", "empty", 
    "unknown", "nil", "undefined", "not available", "no value", "void", 
    "error", "failed", "no * available", "not * found", "unable * process", 
    "service * unavailable", "access * denied", "failed * attempt", "missing * data", 
    "unexpected * error"
]

invalid_df = filter_invalid_rows(df_pin, invalid_values)
display(invalid_df)

category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id,invalid_columns,invalid_values
event-planning,No description available Story format,1,389k,https://i.pinimg.com/videos/thumbnails/originals/2d/7c/24/2d7c242d495928e991c1f8ace72fe327.0000001.jpg,4618,multi-video(story page format),WittyVows,Local save in /data/event-planning,"Desi Wedding Decor,Engagement Decorations,Outdoor Wedding Decorations,Backdrop Decorations,Backdrop Wedding,Mehendi,Mehndi Decor,Indian Wedding Video,Indian Wedding Photos",Top trending colours for Mehendi Décor in 2021,06a9f441-546e-4d9a-916b-aa64a507ca60,List(description),List(no * available)
education,No description available Story format,1,198k,https://i.pinimg.com/videos/thumbnails/originals/a8/53/66/a853664934b9adfe74faa860228a4359.0000001.jpg,4061,multi-video(story page format),CameraMath,Local save in /data/education,"Math Strategies,Math Resources,Math Activities,Math Tips,Math For Kids,Fun Math,Cool Math Tricks,Math Fractions,Multiplication",Mixed number to decimal number🙋‍♂️ Did u get it? #studywithme #cameramathapp,098a8877-cd77-4154-9b8e-fe825c86d93c,List(description),List(no * available)
beauty,No description available Story format,1,18k,https://i.pinimg.com/videos/thumbnails/originals/37/5c/43/375c4345e4f3eabc9cd7e1a82641c6c0.0000001.jpg,1345,multi-video(story page format),Dr. Mark Strom,Local save in /data/beauty,"Skin Care Routine Steps,Skin Routine,Skin Care Tips,Beauty Tips For Glowing Skin,Clear Skin Tips,Healthy Skin Tips,Face Skin Care,Skin Care Treatments,Up Girl",How to make your pores look smaller!,18cb851a-8da2-4ea9-9e65-625f7e917c08,List(description),List(no * available)
travel,No description available Story format,1,8M,https://i.pinimg.com/videos/thumbnails/originals/68/6c/ab/686cabd4f4837aed545c56946ade0786.0000001.jpg,10072,multi-video(story page format),Tastemade,Local save in /data/travel,"Packing Tips For Vacation,Vacation Trips,Vacation Spots,Vacation Travel,Vacation Ideas,Vacations,Amazing Places On Earth,Beautiful Places To Travel,Beautiful Hotels",Glass Castle Theme Park in South Korea,2aafef6f-c0c4-4f4e-8e2d-cd3ccf5fe5e2,List(description),List(no * available)
vehicles,No description available Story format,1,25k,https://i.pinimg.com/videos/thumbnails/originals/a2/9c/7b/a29c7bd22ded68335ab01e508a8e5a16.0000001.jpg,10747,multi-video(story page format),Ricardo Lagreca,Local save in /data/vehicles,"Old Vintage Cars,Antique Cars,Vintage Classic Cars,Classy Cars,Sexy Cars,Ford Classic Cars,Classic Trucks,American Classic Cars,Classic Sports Cars",Volkswagen Classic,2e00a361-fe54-4b79-a377-85268e6002b9,List(description),List(no * available)
vehicles,No description available,1,171k,https://i.pinimg.com/originals/a0/b9/d8/a0b9d873c610a1793b9888dd2bd7075d.jpg,10962,image,vintagetopia,Local save in /data/vehicles,"Chevrolet Camaro 1969,67 Camaro,Chevy Chevelle Ss,Chevy Pickups,Corvette,Ford Mustang 1969,Mustang Cobra,Old Muscle Cars,Chevy Muscle Cars",Chevrolet Camaro 1969,332ce8f1-303d-4692-9d3a-3d630c0b14b6,List(description),List(no * available)
travel,No description available Story format,0,User Info Error,Image src error.,9654,multi-video(story page format),User Info Error,Local save in /data/travel,"N,o, ,T,a,g,s, ,A,v,a,i,l,a,b,l,e",No Title Data Available,4bc2ade8-f6f5-4744-87bf-0eba8f2bc736,"List(description, follower_count, image_src, poster_name, title)","List(no * available, error)"
home-decor,No description available Story format,0,4k,Image src error.,6571,multi-video(story page format),Melissa,Local save in /data/home-decor,"Up House,Cozy House,Living Room Inspiration,Home Decor Inspiration,Decor Ideas,Room Ideas,Boho Living Room,Bohemian Living,Estilo Boho",Spring decor,532b06d4-586a-4020-bced-ef80cb464ac7,"List(description, image_src)","List(no * available, error)"
travel,No description available Story format,1,5k,https://i.pinimg.com/videos/thumbnails/originals/0a/8a/86/0a8a86bcde8d25a

### Replacing empty and/or non relevant data with Nones

In [0]:
invalid_values = {
    "description": ["No description available Story format", "No description available"],
    "follower_count": ["User Info Error"],
    "image_src": ["Image src error."],
    "poster_name": ["User Info Error"],
    "tag_list": ["N,o, ,T,a,g,s, ,A,v,a,i,l,a,b,l,e"],
    "title": ["No Title Data Available"]
}

for column, values in invalid_values.items():
    df_pin = df_pin.withColumn(column, F.when(F.col(column).isin(values), None).otherwise(F.col(column)))

display(df_pin)

category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id
vehicles,"The MRZR Diesel is an ultra-light turbo diesel combat vehicle perfectly suited for a wide range of tactical missions and certified for V-22, H-53, and H-47 internal air transport.",1,1k,https://i.pinimg.com/originals/de/fe/d6/defed65f30023145c4751f7e469334b9.jpg,10345,image,Michael Linderer,Local save in /data/vehicles,"Military Gear,Military Equipment,Army Vehicles,Armored Vehicles,Offroad,Polaris Off Road,Bug Out Vehicle,By Any Means Necessary,Polaris Ranger",Polaris MRZR Diesel | Polaris Government & Defense,00e1f3c5-ef2d-468c-91b1-b1bdc7ea8e37
tattoos,The foot is the sexiest part of the body and it is one of the best part of the body to have a tattoo. If you like to have a personal tattoo on your body then the foot is the bes…,1,873,https://i.pinimg.com/originals/36/e6/11/36e6116b96c87157b934dd9a4ea422e0.jpg,9050,image,Zbear,Local save in /data/tattoos,"Celtic Tattoo For Women,Simple Tattoos For Women,Ankle Tattoos For Women,Tattoos For Women Flowers,Celtic Tattoos,Flower Tattoo Meanings,Flower Tattoo Foot,Flower Tattoos,Tulip Tattoo",27 Outclass Foot Tattoos Ideas For Women,0103fb4b-a758-41da-a11b-234afa68505b
home-decor,CREATE A SPRING INSPIRED SOFA- Part of creating a beautiful home is giving a nod to the season! Sofas can help you breathe spring air into a room!,1,203k,https://i.pinimg.com/originals/49/2d/00/492d00fe990720051e93940e0b2d1daa.jpg,6349,image,StoneGable,Local save in /data/home-decor,"Coffee Table Styling,Decorating Coffee Tables,Coffee Table Tray Decor,Sofa Table Decor,Spring Home Decor,Diy Home Decor,Diy Décoration,Deco Table,Home Decor Inspiration",CREATE A SPRING INSPIRED SOFA,011b2462-ff5d-4b7c-bdeb-37781828e650
art,Shadow drawing is an awesome art process for young artists of any age. Considering shapes and positive & negative space are all part of making shadow art!,1,221k,https://i.pinimg.com/originals/6f/a6/ac/6fa6aca0168d8dee17c07c95210eb43d.png,341,image,The Kitchen Table Classroom,Local save in /data/art,"Shadow Drawing,Shadow Art,Middle School Art,Art School,High School,Programme D'art,Arte Elemental,Classe D'art,Art Curriculum",Shadow Drawing - An Experience in Positive and Negative Shapes - The Kitchen Table Classroom,0143fe12-3325-4e65-b138-f739df3cad77
art,"This fall tree silhouette art project is inspired by the beautiful leaves, as well as our animal friends that are scurrying around this time of year.",1,161k,https://i.pinimg.com/originals/02/c4/55/02c4558478c78e164ec2441fb15258fc.jpg,732,image,Make and Takes,Local save in /data/art,"September Art,September Crafts,Animal Art Projects,Fall Art Projects,Thanksgiving Art Projects,Kids Thanksgiving,Infant Art Projects,Halloween Art Projects,Art Projects",Fall Tree Silhouette Art Project - Make and Takes,02325753-68ad-4692-97a0-5b05bd513a5d
education,A list of more than 36 accommodations and strategies to address anxiety in your IEP or 504. #anxiety #GAD #IEP #school #education #IEPmeetingadvice,1,24k,https://i.pinimg.com/originals/d8/0f/2d/d80f2daa66493f1dc2f7616e5e2d98c1.jpg,3629,image,Fuzzymama | Simple ADHD Parenting Strategies,Local save in /data/education,"Gifted Education,Special Education Classroom,Physics Classroom,Kids Education,Physical Education,Special Education Organization,Texas Education,Education City,Classroom Board",{Anxiety} 43 Helpful IEP and 504 Plan Accommodations • A Day In Our Shoes,02a08753-216a-4107-b6cf-7051bf84a775
beauty,"Lymphatic drainage facial massage can improve uffiness, dull skin, acne, and even sensitivity. And doing it yourself is easier than you think!",1,6k,https://i.pinimg.com/originals/0f/88/3b/0f883b97c5014fe151b00d355800227e.png,1619,image,Renee Rouleau Skin Care,Local save in /data/beauty,"Beauty Care,Beauty Skin,Lymphatic Drainage Massage,Face Massage,Massage Tips,Face Yoga,Massage Techniques,Health And Beauty Tips,Beauty Tips For Skin",Lymphatic Drainage—

Databricks data profile. Run in Databricks to view.

#### Ensuring `image_src` have valid path

In [0]:
df = df_pin.withColumn("image_src", F.when(F.col("image_src").rlike(r"^https://i\.pinimg\.com/.*"), F.col("image_src")).otherwise(None))

display(df_pin)

category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id
vehicles,"The MRZR Diesel is an ultra-light turbo diesel combat vehicle perfectly suited for a wide range of tactical missions and certified for V-22, H-53, and H-47 internal air transport.",1,1k,https://i.pinimg.com/originals/de/fe/d6/defed65f30023145c4751f7e469334b9.jpg,10345,image,Michael Linderer,Local save in /data/vehicles,"Military Gear,Military Equipment,Army Vehicles,Armored Vehicles,Offroad,Polaris Off Road,Bug Out Vehicle,By Any Means Necessary,Polaris Ranger",Polaris MRZR Diesel | Polaris Government & Defense,00e1f3c5-ef2d-468c-91b1-b1bdc7ea8e37
tattoos,The foot is the sexiest part of the body and it is one of the best part of the body to have a tattoo. If you like to have a personal tattoo on your body then the foot is the bes…,1,873,https://i.pinimg.com/originals/36/e6/11/36e6116b96c87157b934dd9a4ea422e0.jpg,9050,image,Zbear,Local save in /data/tattoos,"Celtic Tattoo For Women,Simple Tattoos For Women,Ankle Tattoos For Women,Tattoos For Women Flowers,Celtic Tattoos,Flower Tattoo Meanings,Flower Tattoo Foot,Flower Tattoos,Tulip Tattoo",27 Outclass Foot Tattoos Ideas For Women,0103fb4b-a758-41da-a11b-234afa68505b
home-decor,CREATE A SPRING INSPIRED SOFA- Part of creating a beautiful home is giving a nod to the season! Sofas can help you breathe spring air into a room!,1,203k,https://i.pinimg.com/originals/49/2d/00/492d00fe990720051e93940e0b2d1daa.jpg,6349,image,StoneGable,Local save in /data/home-decor,"Coffee Table Styling,Decorating Coffee Tables,Coffee Table Tray Decor,Sofa Table Decor,Spring Home Decor,Diy Home Decor,Diy Décoration,Deco Table,Home Decor Inspiration",CREATE A SPRING INSPIRED SOFA,011b2462-ff5d-4b7c-bdeb-37781828e650
art,Shadow drawing is an awesome art process for young artists of any age. Considering shapes and positive & negative space are all part of making shadow art!,1,221k,https://i.pinimg.com/originals/6f/a6/ac/6fa6aca0168d8dee17c07c95210eb43d.png,341,image,The Kitchen Table Classroom,Local save in /data/art,"Shadow Drawing,Shadow Art,Middle School Art,Art School,High School,Programme D'art,Arte Elemental,Classe D'art,Art Curriculum",Shadow Drawing - An Experience in Positive and Negative Shapes - The Kitchen Table Classroom,0143fe12-3325-4e65-b138-f739df3cad77
art,"This fall tree silhouette art project is inspired by the beautiful leaves, as well as our animal friends that are scurrying around this time of year.",1,161k,https://i.pinimg.com/originals/02/c4/55/02c4558478c78e164ec2441fb15258fc.jpg,732,image,Make and Takes,Local save in /data/art,"September Art,September Crafts,Animal Art Projects,Fall Art Projects,Thanksgiving Art Projects,Kids Thanksgiving,Infant Art Projects,Halloween Art Projects,Art Projects",Fall Tree Silhouette Art Project - Make and Takes,02325753-68ad-4692-97a0-5b05bd513a5d
education,A list of more than 36 accommodations and strategies to address anxiety in your IEP or 504. #anxiety #GAD #IEP #school #education #IEPmeetingadvice,1,24k,https://i.pinimg.com/originals/d8/0f/2d/d80f2daa66493f1dc2f7616e5e2d98c1.jpg,3629,image,Fuzzymama | Simple ADHD Parenting Strategies,Local save in /data/education,"Gifted Education,Special Education Classroom,Physics Classroom,Kids Education,Physical Education,Special Education Organization,Texas Education,Education City,Classroom Board",{Anxiety} 43 Helpful IEP and 504 Plan Accommodations • A Day In Our Shoes,02a08753-216a-4107-b6cf-7051bf84a775
beauty,"Lymphatic drainage facial massage can improve uffiness, dull skin, acne, and even sensitivity. And doing it yourself is easier than you think!",1,6k,https://i.pinimg.com/originals/0f/88/3b/0f883b97c5014fe151b00d355800227e.png,1619,image,Renee Rouleau Skin Care,Local save in /data/beauty,"Beauty Care,Beauty Skin,Lymphatic Drainage Massage,Face Massage,Massage Tips,Face Yoga,Massage Techniques,Health And Beauty Tips,Beauty Tips For Skin",Lymphatic Drainage—

### Editing `save_location` so it only has the path

In [0]:
df_pin = df_pin.withColumn(
    "save_location",
    F.when(F.col("save_location").startswith("Local save in "), F.regexp_replace(F.col("save_location"), r"^Local save in ", ""))
    .otherwise(None)
)

display(df_pin)

category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id
vehicles,"The MRZR Diesel is an ultra-light turbo diesel combat vehicle perfectly suited for a wide range of tactical missions and certified for V-22, H-53, and H-47 internal air transport.",1,1k,https://i.pinimg.com/originals/de/fe/d6/defed65f30023145c4751f7e469334b9.jpg,10345,image,Michael Linderer,/data/vehicles,"Military Gear,Military Equipment,Army Vehicles,Armored Vehicles,Offroad,Polaris Off Road,Bug Out Vehicle,By Any Means Necessary,Polaris Ranger",Polaris MRZR Diesel | Polaris Government & Defense,00e1f3c5-ef2d-468c-91b1-b1bdc7ea8e37
tattoos,The foot is the sexiest part of the body and it is one of the best part of the body to have a tattoo. If you like to have a personal tattoo on your body then the foot is the bes…,1,873,https://i.pinimg.com/originals/36/e6/11/36e6116b96c87157b934dd9a4ea422e0.jpg,9050,image,Zbear,/data/tattoos,"Celtic Tattoo For Women,Simple Tattoos For Women,Ankle Tattoos For Women,Tattoos For Women Flowers,Celtic Tattoos,Flower Tattoo Meanings,Flower Tattoo Foot,Flower Tattoos,Tulip Tattoo",27 Outclass Foot Tattoos Ideas For Women,0103fb4b-a758-41da-a11b-234afa68505b
home-decor,CREATE A SPRING INSPIRED SOFA- Part of creating a beautiful home is giving a nod to the season! Sofas can help you breathe spring air into a room!,1,203k,https://i.pinimg.com/originals/49/2d/00/492d00fe990720051e93940e0b2d1daa.jpg,6349,image,StoneGable,/data/home-decor,"Coffee Table Styling,Decorating Coffee Tables,Coffee Table Tray Decor,Sofa Table Decor,Spring Home Decor,Diy Home Decor,Diy Décoration,Deco Table,Home Decor Inspiration",CREATE A SPRING INSPIRED SOFA,011b2462-ff5d-4b7c-bdeb-37781828e650
art,Shadow drawing is an awesome art process for young artists of any age. Considering shapes and positive & negative space are all part of making shadow art!,1,221k,https://i.pinimg.com/originals/6f/a6/ac/6fa6aca0168d8dee17c07c95210eb43d.png,341,image,The Kitchen Table Classroom,/data/art,"Shadow Drawing,Shadow Art,Middle School Art,Art School,High School,Programme D'art,Arte Elemental,Classe D'art,Art Curriculum",Shadow Drawing - An Experience in Positive and Negative Shapes - The Kitchen Table Classroom,0143fe12-3325-4e65-b138-f739df3cad77
art,"This fall tree silhouette art project is inspired by the beautiful leaves, as well as our animal friends that are scurrying around this time of year.",1,161k,https://i.pinimg.com/originals/02/c4/55/02c4558478c78e164ec2441fb15258fc.jpg,732,image,Make and Takes,/data/art,"September Art,September Crafts,Animal Art Projects,Fall Art Projects,Thanksgiving Art Projects,Kids Thanksgiving,Infant Art Projects,Halloween Art Projects,Art Projects",Fall Tree Silhouette Art Project - Make and Takes,02325753-68ad-4692-97a0-5b05bd513a5d
education,A list of more than 36 accommodations and strategies to address anxiety in your IEP or 504. #anxiety #GAD #IEP #school #education #IEPmeetingadvice,1,24k,https://i.pinimg.com/originals/d8/0f/2d/d80f2daa66493f1dc2f7616e5e2d98c1.jpg,3629,image,Fuzzymama | Simple ADHD Parenting Strategies,/data/education,"Gifted Education,Special Education Classroom,Physics Classroom,Kids Education,Physical Education,Special Education Organization,Texas Education,Education City,Classroom Board",{Anxiety} 43 Helpful IEP and 504 Plan Accommodations • A Day In Our Shoes,02a08753-216a-4107-b6cf-7051bf84a775
beauty,"Lymphatic drainage facial massage can improve uffiness, dull skin, acne, and even sensitivity. And doing it yourself is easier than you think!",1,6k,https://i.pinimg.com/originals/0f/88/3b/0f883b97c5014fe151b00d355800227e.png,1619,image,Renee Rouleau Skin Care,/data/beauty,"Beauty Care,Beauty Skin,Lymphatic Drainage Massage,Face Massage,Massage Tips,Face Yoga,Massage Techniques,Health And Beauty Tips,Beauty Tips For Skin",Lymphatic Drainage—Could This Be a Game Changer for Your Skin?,02caacde-9d42-4186-a861-5d20db668b70
beauty,"Dealing w

### Editing follower_count so it has numerical values & datatype

In [0]:
df_pin = df_pin.withColumn(
    "follower_count",
    #Check for numeric, k, or m suffix
    F.when(F.col("follower_count").rlike(r"^\d+(\.\d+)?[kKmM]?$"), 
         F.regexp_replace(F.regexp_replace(F.lower(F.col("follower_count")), "k", "000"), "m", "000000").cast("int"))
    .otherwise(None)
)

display(df_pin)

category,description,downloaded,follower_count,image_src,index,is_image_or_video,poster_name,save_location,tag_list,title,unique_id
vehicles,"The MRZR Diesel is an ultra-light turbo diesel combat vehicle perfectly suited for a wide range of tactical missions and certified for V-22, H-53, and H-47 internal air transport.",1,1000,https://i.pinimg.com/originals/de/fe/d6/defed65f30023145c4751f7e469334b9.jpg,10345,image,Michael Linderer,/data/vehicles,"Military Gear,Military Equipment,Army Vehicles,Armored Vehicles,Offroad,Polaris Off Road,Bug Out Vehicle,By Any Means Necessary,Polaris Ranger",Polaris MRZR Diesel | Polaris Government & Defense,00e1f3c5-ef2d-468c-91b1-b1bdc7ea8e37
tattoos,The foot is the sexiest part of the body and it is one of the best part of the body to have a tattoo. If you like to have a personal tattoo on your body then the foot is the bes…,1,873,https://i.pinimg.com/originals/36/e6/11/36e6116b96c87157b934dd9a4ea422e0.jpg,9050,image,Zbear,/data/tattoos,"Celtic Tattoo For Women,Simple Tattoos For Women,Ankle Tattoos For Women,Tattoos For Women Flowers,Celtic Tattoos,Flower Tattoo Meanings,Flower Tattoo Foot,Flower Tattoos,Tulip Tattoo",27 Outclass Foot Tattoos Ideas For Women,0103fb4b-a758-41da-a11b-234afa68505b
home-decor,CREATE A SPRING INSPIRED SOFA- Part of creating a beautiful home is giving a nod to the season! Sofas can help you breathe spring air into a room!,1,203000,https://i.pinimg.com/originals/49/2d/00/492d00fe990720051e93940e0b2d1daa.jpg,6349,image,StoneGable,/data/home-decor,"Coffee Table Styling,Decorating Coffee Tables,Coffee Table Tray Decor,Sofa Table Decor,Spring Home Decor,Diy Home Decor,Diy Décoration,Deco Table,Home Decor Inspiration",CREATE A SPRING INSPIRED SOFA,011b2462-ff5d-4b7c-bdeb-37781828e650
art,Shadow drawing is an awesome art process for young artists of any age. Considering shapes and positive & negative space are all part of making shadow art!,1,221000,https://i.pinimg.com/originals/6f/a6/ac/6fa6aca0168d8dee17c07c95210eb43d.png,341,image,The Kitchen Table Classroom,/data/art,"Shadow Drawing,Shadow Art,Middle School Art,Art School,High School,Programme D'art,Arte Elemental,Classe D'art,Art Curriculum",Shadow Drawing - An Experience in Positive and Negative Shapes - The Kitchen Table Classroom,0143fe12-3325-4e65-b138-f739df3cad77
art,"This fall tree silhouette art project is inspired by the beautiful leaves, as well as our animal friends that are scurrying around this time of year.",1,161000,https://i.pinimg.com/originals/02/c4/55/02c4558478c78e164ec2441fb15258fc.jpg,732,image,Make and Takes,/data/art,"September Art,September Crafts,Animal Art Projects,Fall Art Projects,Thanksgiving Art Projects,Kids Thanksgiving,Infant Art Projects,Halloween Art Projects,Art Projects",Fall Tree Silhouette Art Project - Make and Takes,02325753-68ad-4692-97a0-5b05bd513a5d
education,A list of more than 36 accommodations and strategies to address anxiety in your IEP or 504. #anxiety #GAD #IEP #school #education #IEPmeetingadvice,1,24000,https://i.pinimg.com/originals/d8/0f/2d/d80f2daa66493f1dc2f7616e5e2d98c1.jpg,3629,image,Fuzzymama | Simple ADHD Parenting Strategies,/data/education,"Gifted Education,Special Education Classroom,Physics Classroom,Kids Education,Physical Education,Special Education Organization,Texas Education,Education City,Classroom Board",{Anxiety} 43 Helpful IEP and 504 Plan Accommodations • A Day In Our Shoes,02a08753-216a-4107-b6cf-7051bf84a775
beauty,"Lymphatic drainage facial massage can improve uffiness, dull skin, acne, and even sensitivity. And doing it yourself is easier than you think!",1,6000,https://i.pinimg.com/originals/0f/88/3b/0f883b97c5014fe151b00d355800227e.png,1619,image,Renee Rouleau Skin Care,/data/beauty,"Beauty Care,Beauty Skin,Lymphatic Drainage Massage,Face Massage,Massage Tips,Face Yoga,Massage Techniques,Health And Beauty Tips,Beauty Tips For Skin",Lymphatic Drainage—Could This Be a Game Changer for Your Skin?,02caacde-9d42-4186-a861-5d20db668b70
beaut

Databricks data profile. Run in Databricks to view.

Only missing value in follower count is not an invalid datapoint, but a row where the poster has 0 followers!

### Renaming `index` to `ind`

In [0]:
df_pin = df_pin.withColumnRenamed("index", "ind")
print(df_pin.columns)

['category', 'description', 'downloaded', 'follower_count', 'image_src', 'ind', 'is_image_or_video', 'poster_name', 'save_location', 'tag_list', 'title', 'unique_id']


### Re-ording cols

In [0]:
column_order = [
        "ind",
        "unique_id",
        "title",
        "description",
        "follower_count",
        "poster_name",
        "tag_list",
        "is_image_or_video",
        "image_src",
        "save_location",
        "category"
    ]
df_pin = df_pin.select(column_order)
df_pin = df_pin.orderBy("ind")
display(df_pin)

ind,unique_id,title,description,follower_count,poster_name,tag_list,is_image_or_video,image_src,save_location,category
27,1bc67f67-70f6-4c5b-ae03-d8201f4bb9b7,Bulgarian Artist Makes Incredible Illustrations That Glow From Within,"It doesn't matter if you use a pencil, a crayon or the tip of your nose to create art, it is no small feat to produce something that'll knock everyone's socks off. Some artists…",2000000,Bored Panda,"Outline Drawings,Pencil Art Drawings,Cool Art Drawings,Horse Drawings,Graphite Drawings,Drawings Of Angels,Hair Drawings,Girl Drawing Sketches,Drawing Artist",image,https://i.pinimg.com/originals/0d/e2/ba/0de2ba7b5eaa155211bb2f219fdedf3a.jpg,/data/art,art
88,c213eac9-827e-469b-b890-273fec05f735,"Pink Marble Wall Art, Abstract Marbling, Gold Abstract Wall Art, Gold Marble Print, Marble Artwork Print, Large Wall Art, Marble Wall Decor - 1 Panel 12x9 / Gallery Wrap",Marble Wall Art Modern Abstract Canvas Artwork Contemporary Home Decor Canvas Wall Art Ready to Hang Canvas Each canvas is professionally printed and hand-stretched in the USA.…,305,Wall Canvas Mall,"Pink Canvas Art,Large Canvas Art,Pink Art,Large Wall Art,Wall Canvas,Abstract Canvas,Canvas Artwork,Large Abstract Wall Art,Marble Wall",image,https://i.pinimg.com/originals/24/d6/36/24d636425a74dcb7a19907a8c1f4b37d.jpg,/data/art,art
97,3c48c1da-5ff0-4264-8f1c-24279d983e77,African Sunset Shadow Tracing Art - Taming Little Monsters,This African sunset shadow tracing art is a great actvity for kids. A fun way to use those mini world figures in a new and interesting creative process.,4000,Taming Little Monsters - Fun Activities for Kids,"Kids Crafts,Projects For Kids,Arts And Crafts,Art Crafts,Kids Diy,Children Art Projects,Art For Children,Kids Art Lessons,Decor Crafts",image,https://i.pinimg.com/originals/8f/b3/20/8fb320006114e28437ebbd4d46638bcb.png,/data/art,art
122,a6c2177b-6a7f-4904-8bcc-5b56826b614e,Felt Floral Art That I Made With Hundreds Of Hand-Cut Pieces,"I'm a Brazilian graphic designer and this is my project called Lhama, where I develop art compositions using felt as creative material. Rescuing the experience of working with m…",2000000,Bored Panda,"Arte Floral,Motif Floral,Felt Diy,Felt Crafts,Felt Embroidery,Art Plastique,Doodle Art,Textile Art,Collage Art",image,https://i.pinimg.com/originals/9b/43/4a/9b434a120fddfe6ee28bb50ff18938e8.jpg,/data/art,art
220,224e6074-c7d9-4d2a-be79-d4288404a534,50 Bachelor Pad Wall Art Design Ideas For Men - Cool Visual Decor,Graduate from tacky posters to style and substance with the top 50 best bachelor pad wall art design ideas for men. Explore cool visual decor.,800000,Next Luxury,"Apple Painting,Easy Canvas Painting,Acrylic Canvas,Diy Canvas,Painting & Drawing,Canvas Art,Canvas Ideas,Canvas Walls,Rock Painting",image,https://i.pinimg.com/originals/6b/ed/7b/6bed7b0dd2ef0cc630219b58f8529c1d.jpg,/data/art,art
225,2d27cf57-ae20-4166-9bca-8d04710c69d6,I Capture Dream-Like Images Using A Graphite Pencil,Hi! My name is Miles Johnston. I got into practicing drawing seriously around 11 years ago when I was 13. I was a frequent poster on a forum that was pretty popular back in the…,2000000,Bored Panda,"Drawing Sketches,Art Drawings,Drawing Lips,Photo Manga,Gravure Illustration,Crayon,Surreal Art,Pretty Art,Art Sketchbook",image,https://i.pinimg.com/originals/c2/3f/89/c23f8900b2ddd3d414d29f5c6b318a21.jpg,/data/art,art
289,94a85275-732d-43ff-8717-a8d8a5bfc212,Glue Drawing with Chalk - The Kitchen Table Classroom,Use drizzled glue to provide structure for this fun chalk blending process. #processart #chalkart #pastelart #artforkids,221000,The Kitchen Table Classroom,"Art Projects For Adults,School Art Projects,Fun Art Projects,Art Therapy Projects,Classe D'art,Wal Art,Art Lessons Elementary,Elementary Art Rooms,Art Education Lessons",video,https://i.pinimg.com/videos/thumbnails/originals/f1/d0/91/f1d091d09de76da157dcfaa8f85b070f.0000001.jpg,/data/art,art
318,e479f26c-a702-440e-8df1-7c99ed6aa794,33 MORE Totally Fr

## Geo Topic

### Loading in data

In [0]:
df_geo = get_dataframe_from_drive('038444ac863e.geo')

#### Initial preview of data

In [0]:
display(df_geo.observe)
print(df_geo.count())

<bound method DataFrame.observe of DataFrame[country: string, ind: bigint, latitude: double, longitude: double, timestamp: string]>

372


In [0]:
display(df_geo.head(5))

country,ind,latitude,longitude,timestamp
British Indian Ocean Territory (Chagos Archipelago),4679,-64.3183,43.1505,2018-09-15T22:04:51
British Indian Ocean Territory (Chagos Archipelago),5797,-83.7472,8.65953,2017-12-21T16:57:42
Antarctica (the territory South of 60 deg S),2268,-68.2538,-171.706,2018-05-18T12:52:00
Antarctica (the territory South of 60 deg S),2295,-88.4642,-171.061,2020-09-17T10:30:11
Antarctica (the territory South of 60 deg S),7593,-88.4642,-171.061,2022-07-24T22:57:38


### Removing Duplicates

Not a unique id column here but the index works well for dropping duplicates

In [0]:
print(df_geo.count())
df_geo = df_geo.dropDuplicates(['ind'])
display(df_geo.count())

372


365

### Detecting Null values

In [0]:
null_counts = get_null_counts_df(df_geo)
display(null_counts)

country,ind,latitude,longitude,timestamp
0,0,0,0,0


### Identifying non relavent values through functions instead of Data Profiler

In [0]:
cols_to_ignore = ['timestamp']
filter_mostly_numeric_strings(df_geo, threshold=0.5, exclude_cols=cols_to_ignore)

🔍 Searching for Potential Numeric Issues in Column: country


country,ind,latitude,longitude,timestamp


In [0]:
invalid_values = [
    "none", "null", "n/a", "no data", "missing", "invalid", "empty", 
    "unknown", "nil", "undefined", "not available", "no value", "void", 
    "error", "failed", "no * available", "not * found", "unable * process", 
    "service * unavailable", "access * denied", "failed * attempt", "missing * data", 
    "unexpected * error"
]

invalid_df = filter_invalid_rows(df_geo, invalid_values)
display(invalid_df)

country,ind,latitude,longitude,timestamp,invalid_columns,invalid_values


### Creating `coordinates` col

In [0]:
df_geo = df_geo.withColumn("coordinates", F.array(F.col("latitude"), F.col("longitude")))
# dropping latitude & longitude cols
df_geo = df_geo.drop("latitude","longitude")
display(df_geo.head(5))

country,ind,timestamp,coordinates
Albania,27,2020-11-15T23:51:27,"List(-88.8298, -170.188)"
Aruba,88,2018-05-26T02:45:32,"List(-75.3, -169.77)"
Montserrat,97,2018-12-24T22:15:35,"List(-29.1712, -107.111)"
Albania,122,2018-10-09T05:43:21,"List(-88.8298, -170.188)"
Argentina,220,2022-06-25T02:03:38,"List(-89.4739, -176.154)"


### Correcting `timestamp` col datatype

In [0]:
df_geo = df_geo.withColumn("timestamp", F.to_timestamp("timestamp"))
display(df_geo.head(5))

country,ind,timestamp,coordinates
Albania,27,2020-11-15T23:51:27Z,"List(-88.8298, -170.188)"
Aruba,88,2018-05-26T02:45:32Z,"List(-75.3, -169.77)"
Montserrat,97,2018-12-24T22:15:35Z,"List(-29.1712, -107.111)"
Albania,122,2018-10-09T05:43:21Z,"List(-88.8298, -170.188)"
Argentina,220,2022-06-25T02:03:38Z,"List(-89.4739, -176.154)"


### Re-ording cols

In [0]:
column_order = [
        "ind",
        "country",
        "coordinates",
        "timestamp",
    ]
df_geo = df_geo.select(column_order)
df_geo = df_geo.orderBy("ind")
display(df_geo)

ind,country,coordinates,timestamp
27,Albania,"List(-88.8298, -170.188)",2020-11-15T23:51:27Z
88,Aruba,"List(-75.3, -169.77)",2018-05-26T02:45:32Z
97,Montserrat,"List(-29.1712, -107.111)",2018-12-24T22:15:35Z
122,Albania,"List(-88.8298, -170.188)",2018-10-09T05:43:21Z
220,Argentina,"List(-89.4739, -176.154)",2022-06-25T02:03:38Z
225,Albania,"List(-88.8298, -170.188)",2019-02-07T11:02:33Z
289,Bahamas,"List(-87.0574, -164.826)",2022-03-07T03:54:05Z
318,Costa Rica,"List(82.0824, 34.2465)",2018-11-20T06:26:06Z
338,China,"List(-60.4435, -145.379)",2020-04-05T06:35:26Z
341,Bahamas,"List(-87.0574, -164.826)",2020-08-27T20:02:06Z


## User Topic

### Loading in data

In [0]:
df_user = get_dataframe_from_drive('038444ac863e.user')

#### Initial preview of data

In [0]:
display(df_user.observe)
print(df_user.count())
display(df_user.head(5))

<bound method DataFrame.observe of DataFrame[age: bigint, date_joined: string, first_name: string, ind: bigint, last_name: string]>

372


age,date_joined,first_name,ind,last_name
33,2016-08-12T07:57:25,Christopher,3860,Richardson
37,2016-08-15T22:17:23,Christopher,5831,Martinez
20,2015-10-23T04:13:23,Alexandria,3640,Alvarado
20,2015-10-23T04:13:23,Alexandria,3899,Alvarado
23,2015-10-31T19:20:09,Alexandria,6185,Anderson


### Removing Duplicates

Not a unique id column here but the index works well for dropping duplicates

In [0]:
print(df_user.count())
df_user = df_user.dropDuplicates(['ind'])
display(df_user.count())

372


365

### Detecting Null values

In [0]:
null_counts = get_null_counts_df(df_user)
display(null_counts)

age,date_joined,first_name,ind,last_name
0,0,0,0,0


### Identifying non relavent values through functions instead of Data Profiler

In [0]:
cols_to_ignore = ['date_joined']
filter_mostly_numeric_strings(df_user, threshold=0.5, exclude_cols=cols_to_ignore)

🔍 Searching for Potential Numeric Issues in Column: first_name


age,date_joined,first_name,ind,last_name


🔍 Searching for Potential Numeric Issues in Column: last_name


age,date_joined,first_name,ind,last_name


In [0]:
invalid_values = [
    "none", "null", "n/a", "no data", "missing", "invalid", "empty", 
    "unknown", "nil", "undefined", "not available", "no value", "void", 
    "error", "failed", "no * available", "not * found", "unable * process", 
    "service * unavailable", "access * denied", "failed * attempt", "missing * data", 
    "unexpected * error"
]

invalid_df = filter_invalid_rows(df_user, invalid_values)
display(invalid_df)

age,date_joined,first_name,ind,last_name,invalid_columns,invalid_values


### Making `user_name` col

In [0]:
df_user = df_user.withColumn("user_name", F.concat(F.col("first_name"), F.lit(" "), F.col("last_name")))
df_user = df_user.drop("first_name", "last_name")
display(df_user.head(5))

age,date_joined,ind,user_name
20,2015-10-21T21:26:45,27,Adam Acosta
24,2015-11-09T03:47:45,88,Carlos Dixon
32,2016-03-10T04:11:31,97,Brittany Butler
20,2015-10-21T21:26:45,122,Adam Acosta
23,2015-11-28T11:52:37,220,Andrew Anderson


### Correcting `date_joined` col datatype

In [0]:
df_user = df_user.withColumn("date_joined", F.to_timestamp("date_joined"))
display(df_user.head(5))

age,date_joined,ind,user_name
20,2015-10-21T21:26:45Z,27,Adam Acosta
24,2015-11-09T03:47:45Z,88,Carlos Dixon
32,2016-03-10T04:11:31Z,97,Brittany Butler
20,2015-10-21T21:26:45Z,122,Adam Acosta
23,2015-11-28T11:52:37Z,220,Andrew Anderson
